# AeroSynth-Eval — Qwen2.5-VL-3B on a free Colab GPU
Runs the frozen 12-case development queue. It does not touch the protected test set.

In [ ]:
import subprocess
gpu = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    text=True,
).strip()
print('GPU:', gpu)

In [ ]:
from pathlib import Path
ROOT = Path('/content/AeroSynth-Eval')
if not ROOT.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'fix/kaggle-p100-pascal',
        'https://github.com/triasha72/AeroSynth-Eval.git', str(ROOT),
    ], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', 'fix/kaggle-p100-pascal'], cwd=ROOT, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/fix/kaggle-p100-pascal'], cwd=ROOT, check=True)
subprocess.run([
    'python', '-m', 'pip', 'install', '-q',
    'transformers>=4.50,<5', 'accelerate', 'qwen-vl-utils', 'torch', 'torchvision',
], check=True)

In [ ]:
OUTPUT = ROOT / 'reports/qwen25_vl_3b_development.json'
if OUTPUT.exists():
    OUTPUT.unlink()
subprocess.run([
    'python', 'scripts/run_qwen25_vl.py', '--output', str(OUTPUT),
], cwd=ROOT, check=True)

In [ ]:
import hashlib, json
payload = json.loads(OUTPUT.read_text())
summary = {key: payload[key] for key in [
    'model_id', 'device', 'case_count', 'parse_rate', 'accuracy', 'mean_latency_ms'
]}
summary['sha256'] = hashlib.sha256(OUTPUT.read_bytes()).hexdigest()
print(json.dumps(summary, indent=2))
from google.colab import files
files.download(str(OUTPUT))